In [ ]:
# %pip install requests tqdm beautifulsoup4

import urllib.request
import csv
import glob
import time
import pandas as pd
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
from bs4 import BeautifulSoup
from io import StringIO
import numpy as np
import os
from tqdm import tqdm
pd.set_option('display.max_columns', 999)
pd.set_option('display.max_rows', 50)

# Scrape Fanfooty data
This notebook is used to scrape the following data from fanfooty:

1. Current player list
2. Match stats for each player
3. Match results/fixture

## 2. Match stats for each player

### Scrape match files from Fanfooty website

In [21]:
# Generate a timestamp for the destination folder
timestr = time.strftime("%Y%m%d-%H%M%S")
destination = "exports/scrape_{}".format(timestr)

# Create the destination directory if it doesn't exist
if not os.path.exists(destination):
    os.mkdir(destination)

    
# Saved: 9201.txt THIS IS A GAME IN OPENING ROUND 2025 THAT WAS PUSHED BACK
# Saved: 9203.txt THIS IS A GAME IN OPENING ROUND 2025 THAT WAS PUSHED BACK
    
start_match = 9345
end_match = 9362 # Adjust these values to the range of match IDs you want to scrape

matches = list(range(start_match, end_match + 1))

# Suppress only the single InsecureRequestWarning from urllib3 needed
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

def return_list_of_urls(match_id):
    full_url_list = []
    for match in match_id:
        url = "https://www.fanfooty.com.au/live/"
        # url = "https://www.fanfooty.com.au/game/direct.php?id="
        extension = ".txt"
        full_url = "{}{}{}".format(url, match, extension)
        full_url_list.append(full_url)
    return full_url_list

list_of_urls = return_list_of_urls(matches)

url_headers = {'User-Agent': 'Mozilla/5.0'}  # Define your headers

for url in list_of_urls:
    try:
        response = requests.get(url, headers=url_headers, verify=False)
        response.raise_for_status()  # Check if the request was successful
        webContent = response.text
        filename = url.split('/')[-1]
        file_path = f"inputs/All Match Data/{filename}"
        
        # Check if the file already exists
        if not os.path.exists(file_path):
            with open(file_path, 'w', encoding="utf-8") as f:
                f.write(webContent)
            print(f"Saved: {filename}")
        else:
            print(f"File already exists: {filename}")
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch {url}: {e}")

File already exists: 9345.txt
File already exists: 9346.txt
File already exists: 9347.txt
File already exists: 9348.txt
File already exists: 9349.txt
File already exists: 9350.txt
File already exists: 9351.txt
File already exists: 9352.txt
File already exists: 9353.txt
File already exists: 9354.txt
File already exists: 9355.txt
File already exists: 9356.txt
File already exists: 9357.txt
File already exists: 9358.txt
File already exists: 9359.txt
File already exists: 9360.txt
File already exists: 9361.txt
File already exists: 9362.txt


### Headers of each field in match file

In [56]:
column_header_names = [
    'Fanfooty Match ID',
    'Fanfooty Match URL',
    'Round',
    'Year',
    'Player ID',
    'First Name',
    'Surname',
    'Team',
    'null',
    'DT',
    'SC',
    'null2',
    'null3',
    'null4',
    'Kicks',
    'Handballs',
    'Marks',
    'Tackles',
    'Hitouts',
    'Frees for',
    'Frees against',
    'Goals',
    'Behinds',
    'Not sure',
    'Tag',
    'Tag Notes',
    'Tag 2',
    'Tag 2 Notes',
    'null5',
    'null6',
    'null7',
    'null8',
    'Position',
    'Jumper Number',
    'null9',
    'null10',
    'null11',
    'DT own %',
    'SC own %',
    'AF own %',
    'null12',
    'AF Breakeven',
    'null13',
    'Contested Possessions',
    'Clearances',
    'Clangers',
    'Disposal efficiency',
    'Time on ground',
    'Metres gained',
    'Bench status',
    'ExtraField1',
    'ExtraField2',
    'ExtraField3',
    'ExtraField4',
    'ExtraField5',
    'ExtraField6',
    'ExtraField7',
    'ExtraField8',
    'ExtraField9',
    'ExtraField10',
    'ExtraField11',
    'ExtraField12',
    'ExtraField13',
    'ExtraField14',
    'ExtraField15',
    'ExtraField16',
    'ExtraField17',
    'ExtraField18',
    'ExtraField19'

]

### Read match files and write to csv

In [59]:
df_fanfooty_player_raw = pd.DataFrame()
def get_number_of_lines_in_file(data):
    return len(data.split('\n'))

def get_match_id(data):
    name = data.split('\n', 1)[0]
    return name[-9:-5]

def get_url_of_match(data):
    name = data.split('\n', 1)[0]
    url = "http://live.fanfooty.com.au/game/matchcentre.html?id=" + name[-9:-5]
    return url

def get_round(data):
    line = data.split('\n', 1)[1]
    stripped_line = [x.strip() for x in line.split(',')]
    afl_round = stripped_line[4]
    return afl_round

def get_year(data):
    second_line = data.splitlines()[2]
    stripped_second_line = [x.strip() for x in second_line.split(',')]
    afl_year = stripped_second_line[1]
    return afl_year

def get_match_data_list():
    data_list = []
    # path = "inputs/All Match Data/*.txt"
    # path = "inputs/All Match Data/9345.txt"
    path = "inputs/All Match Data/9362.txt"

    for item in glob.glob(path):
        file = open(item, 'r')
        name = file.name
        data = file.read()
        data_list.append(name + '\n' + data)
    return data_list

def return_player_match_data(data_list):
    player_data_for_match = []

    for match in data_list:
        match = os.linesep.join([s for s in match.splitlines() if s])
        number_of_lines = get_number_of_lines_in_file(match)
        afl_round = get_round(match)
        afl_year = get_year(match)
        name = get_url_of_match(match)
        match_id = get_match_id(match)

        for line in range(5, number_of_lines - 1):
            line_data = match.splitlines()[line]
            line_data = [x.strip() for x in line_data.split(',')]
            line_data = [match_id] + [name] + [afl_round] + [afl_year] + line_data
            player_data_for_match.append(line_data)
    return player_data_for_match

match_data_list = get_match_data_list()
player_data = return_player_match_data(match_data_list)
# display(player_data[:5])  # Display the first 5 rows of player data for verification
df_fanfooty_player_raw = pd.DataFrame(player_data, columns=column_header_names)
df_fanfooty_player_raw
    # df.to_csv(f, index=False)

# df_fanfooty_player_raw = pd.read_csv("{}/{}".format(destination, file_name), on_bad_lines='skip')
# df_fanfooty_player_raw.to_csv("kek.csv", index=False)  # Save to a CSV file for verification





# file_name = "fanfooty_match_data_{}.csv".format(timestr)
# with open("{}/{}".format(destination, file_name), "w", newline='') as f:
#     writer = csv.writer(f)
#     writer.writerow(column_header_names)
#     for item in player_data:
#         writer.writerow(item)

# df_fanfooty_player_raw = pd.read_csv("{}/{}".format(destination, file_name), on_bad_lines='skip')
# # df_fanfooty_player_raw.to_csv("kek.csv", index=False)  # Save to a CSV file for verification
# df_fanfooty_player_raw

,Fanfooty Match ID,Fanfooty Match URL,Round,Year,Player ID,First Name,Surname,Team,null,DT,SC,null2,null3,null4,Kicks,Handballs,Marks,Tackles,Hitouts,Frees for,Frees against,Goals,Behinds,Not sure,Tag,Tag Notes,Tag 2,Tag 2 Notes,null5,null6,null7,null8,Position,Jumper Number,null9,null10,null11,DT own %,SC own %,AF own %,null12,AF Breakeven,null13,Contested Possessions,Clearances,Clangers,Disposal efficiency,Time on ground,Metres gained,Bench status,ExtraField1,ExtraField2,ExtraField3,ExtraField4,ExtraField5,ExtraField6,ExtraField7,ExtraField8,ExtraField9,ExtraField10,ExtraField11,ExtraField12,ExtraField13,ExtraField14,ExtraField15,ExtraField16,ExtraField17,ExtraField18,ExtraField19
0,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,296420,Alex,Neal-Bullen,AD,37,135,155,170,111,145,20,11,7,4,0,3,2,3,1,Full Time,star,%s from %P and %M plus %T... aided by %4FF,wing,Playing a HFF role,,0,,,Forward,28,21,78.38,0,0.00,0.09,0.09,,,,9,2,4,80,84,539,0,17,19,30,38,38,39,50,59,2,12,0,2,1,1,6,16,3,1,0
1,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,293222,Rory,Laird,AD,20,104,93,136,83,108,17,6,11,2,0,0,0,0,0,Full Time,hot,%P including %K... also %M and %T,guard,At half back,,0,,,Midfielder,29,19,106.16,0,0.00,9.53,11.99,,,,4,2,3,82,85,243,0,26,28,25,20,22,14,31,31,0,3,0,1,0,0,11,13,2,2,1
2,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,990882,Wayne,Milera,AD,20,101,97,132,82,102,17,4,12,1,0,2,0,0,0,Full Time,hot,%P including %K... also %M,guard,Floating across half back,,0,,,Forward,30,18,73.83,0,0.00,0.28,0.32,,,,2,1,2,81,73,461,1,32,23,16,27,18,17,35,30,2,6,0,4,0,0,12,14,3,0,0
3,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,1011981,Josh,Worrell,AD,24,99,116,127,84,108,17,8,11,0,0,2,1,0,0,Full Time,job,%P and %M... Playing key defender on Long,,,,,,,,24,0,0,0,0.04,0.16,0.19,,,,7,0,4,88,94,409,0,32,33,30,31,7,15,30,37,0,5,0,2,0,0,11,16,1,0,1
4,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,1006013,James,Peatling,AD,26,97,89,127,70,96,15,6,3,6,0,1,0,1,0,Full Time,tagger,%s from %P and %M plus %T... Tagging Rowell,,,,,,,Forward,25,0,0,0,0.09,0.03,0.02,,,,9,7,2,52,84,428,0,27,28,26,23,15,18,29,20,0,6,0,4,1,0,3,5,8,2,0
5,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,1017109,Jake,Soligo,AD,25,95,87,121,81,111,10,14,6,5,0,2,3,1,0,Full Time,shovel,%P and %M plus %T... %s as well... conceded %F...,,,,,,,Midfielder,14,0,0,0,0.40,0.39,0.57,,,,9,3,5,75,75,232,0,14,13,25,24,33,24,23,26,0,6,0,2,1,0,6,7,2,1,0
6,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,1024023,Daniel,Curtin,AD,20,93,132,120,72,94,15,5,6,3,0,1,0,1,1,Full Time,wing,%P including %K... also %M and %T... and kicke...,,,,,,,Back,6,0,0,0,56.86,55.26,52.12,,,,11,2,2,80,87,336,0,38,50,4,14,24,36,27,32,1,6,0,1,1,2,4,11,2,2,0
7,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,992242,Jordan,Dawson,AD,21,88,88,113,69,94,11,9,4,5,0,1,1,1,1,Full Time,shovel,%P and %M plus %T... %s as well... Playing cen...,,,,,,,Back,12,0,0,0,17.40,17.76,12.23,,,,7,5,5,65,76,367,1,24,17,23,24,26,26,15,21,1,6,0,5,0,0,4,7,2,2,0
8,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,1015370,Max,Michalanney,AD,15,84,95,113,72,99,16,6,9,2,0,1,4,0,0,Full Time,guard,%P including %K... also %M and %T... not helpe...,,,,,,,Back,16,0,0,0,0.40,0.45,0.58,,,,6,1,6,86,91,329,0,19,19,27,30,13,18,25,28,0,4,0,1,0,0,9,14,1,1,0
9,9362,http://live.fanfooty.com.au/game/matchcentre.h...,R19,2025,1012807,Sam,Berry,AD,24,82,96,110,55,75,13,2,1,7,0,2,0,1,0,Full Time,dizzy,Friendly fire from Murray in Q3 saw him off fo...,shovel,Rotating through midfield,,,,,Midfielder,3,0,0,0,0.49,0.74,0.09,,,,7,4,3,46,70,366,0,23,22,46,48,3,14,10,12,1,4,0,6,0,0,1,5,5,3,0


### Clean player data

In [7]:
df_fanfooty_player_raw['SC'] = pd.to_numeric(df_fanfooty_player_raw['SC'], errors='coerce')
df_fanfooty_player_raw = df_fanfooty_player_raw.dropna(subset=['SC'])
df_fanfooty_player_raw['SC'] = df_fanfooty_player_raw['SC'].astype('int64')

C:\Users\Richa\AppData\Local\Temp\ipykernel_16904\1527903460.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fanfooty_player_raw['SC'] = df_fanfooty_player_raw['SC'].astype('int64')


### Identify when players were injured during a match
Fanfooty has amazing "tags" that can be used to identify when a player has been injured during a match

If they have certain tags (e.g. concussed) and score below 80 supercoach points, they are judged as injured.

In [8]:
injured_tags = [
    'sore',
    'injured',
    'longterminjured',
    'concussed',
    'heart',
    'subbed'
]

def get_injured_status(row):
#     if (row['Tag'] in injured_tags or row['Tag 2'] in injured_tags) and row['SC'] < 80:
    if (row['Tag'] in injured_tags or row['Tag 2'] in injured_tags):
        return True
    else:
        return False

df_fanfooty_player_raw['Injured'] = df_fanfooty_player_raw.apply(lambda row: get_injured_status(row), axis=1)
df_fanfooty_player_raw

C:\Users\Richa\AppData\Local\Temp\ipykernel_16904\436662756.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fanfooty_player_raw['Injured'] = df_fanfooty_player_raw.apply(lambda row: get_injured_status(row), axis=1)


,Fanfooty Match ID,Fanfooty Match URL,Round,Year,Player ID,First Name,Surname,Team,null,DT,SC,null2,null3,null4,Kicks,Handballs,Marks,Tackles,Hitouts,Frees for,Frees against,Goals,Behinds,Not sure,Tag,Tag Notes,Tag 2,Tag 2 Notes,null5,null6,null7,null8,Position,Jumper Number,null9,null10,null11,DT own %,SC own %,AF own %,null12,AF Breakeven,null13,Contested Possessions,Clearances,Clangers,Disposal efficiency,Time on ground,Metres gained,Bench staus,Injured
0,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,990020.0,Andrew,Embley,WC,30.0,111.0,98,144.0,79.0,112.0,20.0,8.0,1.0,6.0,1.0,1.0,0.0,1.0,0.0,Full Time,gun,Dempsey going with him... %s from %O and %T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,230254.0,Adam,Selwood,WC,50.0,107.0,107,143.0,79.0,108.0,10.0,9.0,4.0,11.0,0.0,3.0,2.0,1.0,0.0,Full Time,hot,Tagged by Lonergan... %D and %M with %T plus %s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,200112.0,Dean,Cox,WC,27.0,99.0,118,114.0,88.0,106.0,9.0,10.0,2.0,2.0,30.0,4.0,1.0,1.0,1.0,Full Time,news,%H and %P with %s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,240016.0,Beau,Waters,WC,26.0,98.0,84,130.0,79.0,117.0,15.0,13.0,5.0,6.0,0.0,0.0,4.0,0.0,0.0,Full Time,news,%P and %M with %F... clangers and FA dampening...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,261911.0,Brad,Ebert,WC,26.0,94.0,109,121.0,70.0,96.0,12.0,9.0,3.0,6.0,0.0,1.0,0.0,1.0,0.0,Full Time,news,Matched up on Winderlich... %D and %T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136963,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1011985.0,Hugo,Ralphsmith,RI,8.0,44.0,51,36.0,30.0,42.0,6.0,2.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0,Full Time,wing,%P including %K... also %T and %M... Starting ...,NaN,NaN,NaN,NaN,NaN,NaN,Back,13.0,0.0,0.0,0.0,0.13,0.44,0.43,NaN,NaN,NaN,2.0,0.0,1.0,87.0,88.0,196.0,0.0,False
136964,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1028521.0,Luke,Trainor,RI,7.0,43.0,63,28.0,39.0,54.0,6.0,9.0,3.0,0.0,0.0,1.0,1.0,0.0,0.0,Full Time,wing,%M and %P... Starting on a wing,NaN,NaN,NaN,NaN,NaN,NaN,Back,31.0,0.0,0.0,0.0,0.00,0.00,0.00,NaN,NaN,NaN,5.0,1.0,3.0,93.0,78.0,205.0,1.0,False
136965,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1017086.0,Tom,Brown,RI,3.0,40.0,64,26.0,29.0,39.0,7.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,Full Time,guard,%P including %K... also %M and %T... Starting ...,NaN,NaN,NaN,NaN,NaN,NaN,Back,30.0,0.0,0.0,0.0,1.55,1.54,0.25,NaN,NaN,NaN,2.0,0.0,0.0,75.0,84.0,214.0,0.0,False
136966,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1023056.0,Steely,Green,RI,8.0,32.0,63,28.0,27.0,36.0,3.0,4.0,1.0,2.0,0.0,1.0,1.0,1.0,0.0,Full Time,wing,%P and %T plus %s... Rotating at half forward,NaN,NaN,NaN,NaN,NaN,NaN,Midfielder,48.0,0.0,0.0,0.0,0.18,0.07,0.22,NaN,NaN,NaN,4.0,0.0,1.0,85.0,84.0,125.0,0.0,False


# Get standard team name

### Get the total SuperCoach and AFL Fantasy scores for each team, for every match

In [9]:
df_match_summary = pd.pivot_table(df_fanfooty_player_raw, index=['Fanfooty Match ID'], values=['SC'], columns=['Team'], aggfunc=np.sum)
df_match_summary = df_match_summary.reset_index()
df_match_summary

C:\Users\Richa\AppData\Local\Temp\ipykernel_16904\3719532010.py:1: FutureWarning: The provided callable <function sum at 0x0000022E1905BC40> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df_match_summary = pd.pivot_table(df_fanfooty_player_raw, index=['Fanfooty Match ID'], values=['SC'], columns=['Team'], aggfunc=np.sum)


Fanfooty Match ID      SC                                              \
Team                        AD      BL      CA      CO      ES  FR      GC   
0                 3425     NaN     NaN     NaN     NaN  1525.0 NaN     NaN   
1                 3426     NaN     NaN     NaN     NaN     NaN NaN     NaN   
2                 3427  1513.0     NaN  1788.0     NaN     NaN NaN     NaN   
3                 3428     NaN     NaN     NaN  1873.0     NaN NaN     NaN   
4                 3429     NaN  1781.0     NaN     NaN     NaN NaN     NaN   
...                ...     ...     ...     ...     ...     ...  ..     ...   
3141              9324     NaN     NaN  1620.0     NaN     NaN NaN     NaN   
3142              9325     NaN     NaN     NaN     NaN     NaN NaN     NaN   
3143              9326     NaN     NaN     NaN  1721.0     NaN NaN     NaN   
3144              9327     NaN     NaN     NaN     NaN     NaN NaN  1648.0   
3145              9328     NaN     NaN     NaN     NaN     NaN NaN     NaN   

                                                                              \
Team  GE      HW  ME      NM      PA      RI      SK      SY      WB      WC   
0    NaN     NaN NaN     NaN     NaN     NaN     NaN     NaN     NaN  1739.0   
1    NaN     NaN NaN  1504.0     NaN     NaN     NaN  1744.0     NaN     NaN   
2    NaN     NaN NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
3    NaN  1423.0 NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
4    NaN     NaN NaN     NaN     NaN     NaN     NaN     NaN  1512.0     NaN   
...   ..     ...  ..     ...     ...     ...     ...     ...     ...     ...   
3141 NaN     NaN NaN  1637.0     NaN     NaN     NaN     NaN     NaN     NaN   
3142 NaN     NaN NaN     NaN  1543.0     NaN     NaN  1750.0     NaN     NaN   
3143 NaN     NaN NaN     NaN     NaN     NaN  1583.0     NaN     NaN     NaN   
3144 NaN     NaN NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
3145 NaN     NaN NaN     NaN     NaN  1369.0     NaN     NaN  1922.0     NaN   

              
Team      WS  
0        NaN  
1        NaN  
2        NaN  
3        NaN  
4        NaN  
...      ...  
3141     NaN  
3142     NaN  
3143     NaN  
3144  1649.0  
3145     NaN  

[3146 rows x 19 columns]

In [10]:
# Create a summary to get the total SC points for each match and team
df_match_summary = pd.pivot_table(df_fanfooty_player_raw, index=['Fanfooty Match ID', 'Team'], values=['SC'], aggfunc=np.sum)
df_match_summary = df_match_summary.reset_index()
df_match_summary['Match_Team_ID'] = df_match_summary['Fanfooty Match ID'].astype('str') + '_' + df_match_summary['Team'].astype('str')
df_match_summary


C:\Users\Richa\AppData\Local\Temp\ipykernel_16904\397094662.py:2: FutureWarning: The provided callable <function sum at 0x0000022E1905BC40> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df_match_summary = pd.pivot_table(df_fanfooty_player_raw, index=['Fanfooty Match ID', 'Team'], values=['SC'], aggfunc=np.sum)


,Fanfooty Match ID,Team,SC,Match_Team_ID
0,3425,ES,1525,3425_ES
1,3425,WC,1739,3425_WC
2,3426,NM,1504,3426_NM
3,3426,SY,1744,3426_SY
4,3427,AD,1513,3427_AD
...,...,...,...,...
6287,9326,SK,1583,9326_SK
6288,9327,GC,1648,9327_GC
6289,9327,WS,1649,9327_WS
6290,9328,RI,1369,9328_RI


In [ ]:
def get_match_team_sc_values(row):
    match_id = row['Fanfooty Match ID']
    my_team = row['Team']
    team_sc = df_match_summary.loc[(df_match_summary['Fanfooty Match ID'] == match_id) & (df_match_summary['Team'] == my_team), 'SC']
    return team_sc.values[0] if not team_sc.empty else np.nan

def get_opposition_team_sc_values(row):
    match_id = row['Fanfooty Match ID']
    my_team = row['Team']
    opposition_team_sc =  df_match_summary.loc[(df_match_summary['Fanfooty Match ID'] == match_id) & (df_match_summary['Team'] != my_team), 'SC']
    return opposition_team_sc.values[0] if not opposition_team_sc.empty else np.nan

# Apply the function with a progress bar
tqdm.pandas()
df_fanfooty_player_raw['Opposition_team_SC'] = df_fanfooty_player_raw.progress_apply(lambda row: get_opposition_team_sc_values(row), axis=1)
df_fanfooty_player_raw['My_team_SC'] = df_fanfooty_player_raw.progress_apply(lambda row: get_match_team_sc_values(row), axis=1)


# Save the DataFrame to a CSV file
# df_fanfooty_player_raw.to_csv(f'df_fanfooty_player_raw_{timestr}.csv')

df_fanfooty_player_raw

100%|██████████| 136675/136675 [01:32<00:00, 1473.08it/s]
C:\Users\Richa\AppData\Local\Temp\ipykernel_16904\2293948667.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fanfooty_player_raw['Opposition_team_SC'] = df_fanfooty_player_raw.progress_apply(lambda row: get_opposition_team_sc_values(row), axis=1)
100%|██████████| 136675/136675 [01:23<00:00, 1628.92it/s]
C:\Users\Richa\AppData\Local\Temp\ipykernel_16904\2293948667.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fanfooty_player_raw['My_

,Fanfooty Match ID,Fanfooty Match URL,Round,Year,Player ID,First Name,Surname,Team,null,DT,SC,null2,null3,null4,Kicks,Handballs,Marks,Tackles,Hitouts,Frees for,Frees against,Goals,Behinds,Not sure,Tag,Tag Notes,Tag 2,Tag 2 Notes,null5,null6,null7,null8,Position,Jumper Number,null9,null10,null11,DT own %,SC own %,AF own %,null12,AF Breakeven,null13,Contested Possessions,Clearances,Clangers,Disposal efficiency,Time on ground,Metres gained,Bench staus,Injured,Opposition_team_SC,My_team_SC
0,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,990020.0,Andrew,Embley,WC,30.0,111.0,98,144.0,79.0,112.0,20.0,8.0,1.0,6.0,1.0,1.0,0.0,1.0,0.0,Full Time,gun,Dempsey going with him... %s from %O and %T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739
1,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,230254.0,Adam,Selwood,WC,50.0,107.0,107,143.0,79.0,108.0,10.0,9.0,4.0,11.0,0.0,3.0,2.0,1.0,0.0,Full Time,hot,Tagged by Lonergan... %D and %M with %T plus %s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739
2,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,200112.0,Dean,Cox,WC,27.0,99.0,118,114.0,88.0,106.0,9.0,10.0,2.0,2.0,30.0,4.0,1.0,1.0,1.0,Full Time,news,%H and %P with %s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739
3,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,240016.0,Beau,Waters,WC,26.0,98.0,84,130.0,79.0,117.0,15.0,13.0,5.0,6.0,0.0,0.0,4.0,0.0,0.0,Full Time,news,%P and %M with %F... clangers and FA dampening...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739
4,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,261911.0,Brad,Ebert,WC,26.0,94.0,109,121.0,70.0,96.0,12.0,9.0,3.0,6.0,0.0,1.0,0.0,1.0,0.0,Full Time,news,Matched up on Winderlich... %D and %T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136963,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1011985.0,Hugo,Ralphsmith,RI,8.0,44.0,51,36.0,30.0,42.0,6.0,2.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0,Full Time,wing,%P including %K... also %T and %M... Starting ...,NaN,NaN,NaN,NaN,NaN,NaN,Back,13.0,0.0,0.0,0.0,0.13,0.44,0.43,NaN,NaN,NaN,2.0,0.0,1.0,87.0,88.0,196.0,0.0,False,1922,1369
136964,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1028521.0,Luke,Trainor,RI,7.0,43.0,63,28.0,39.0,54.0,6.0,9.0,3.0,0.0,0.0,1.0,1.0,0.0,0.0,Full Time,wing,%M and %P... Starting on a wing,NaN,NaN,NaN,NaN,NaN,NaN,Back,31.0,0.0,0.0,0.0,0.00,0.00,0.00,NaN,NaN,NaN,5.0,1.0,3.0,93.0,78.0,205.0,1.0,False,1922,1369
136965,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1017086.0,Tom,Brown,RI,3.0,40.0,64,26.0,29.0,39.0,7.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,Full Time,guard,%P including %K... also %M and %T... Starting ...,NaN,NaN,NaN,NaN,NaN,NaN,Back,30.0,0.0,0.0,0.0,1.55,1.54,0.25,NaN,NaN,NaN,2.0,0.0,0.0,75.0,84.0,214.0,0.0,False,1922,1369
136966,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1023056.0,Steely,Green,RI,8.0,32.0,63,28.0,27.0,36.0,3.0,4.0,1.0,2.0,0.0,1.0,1.0,1.0,0.0,Full Time,wing,%P and %T plus %s... Rotating at half forward,NaN,NaN,NaN,NaN,NaN,NaN,Midfielder,48.0,0.0,0.0,0.0,0.18,0.07,0.22,NaN,NaN,NaN,4.0,0.0,1.0,85.0,84.0,125.0,0.0,False,1922,1369


### Assign an order to rounds

In [12]:
# Create a dictionary to map round names to their order
round_order_dict = {
    'R0': 1, 'R1': 2, 'R2': 3, 'R3': 4, 'R4': 5, 'R5': 6, 'R6': 7, 'R7': 8, 'R8': 9, 'R9': 10,
    'R10': 11, 'R11': 12, 'R12': 13, 'R13': 14, 'R14': 15, 'R15': 16, 'R16': 17, 'R17': 18, 'R18': 19,
    'R19': 20, 'R20': 21, 'R21': 22, 'R22': 23, 'R23': 24, 'R24': 25, 'R25': 26, 'R26': 27, 'R27': 28,
    'R28': 29, 'EF': 30, 'QF': 31, 'SF': 32, 'PF': 33, 'GF': 34
}

# Assuming df_fanfooty_player_raw is your DataFrame
# Create a copy of the DataFrame to avoid the SettingWithCopyWarning
df_fanfooty_player_raw = df_fanfooty_player_raw.copy()

# Assign the 'Round' column with the order using the dictionary
df_fanfooty_player_raw.loc[:, 'Round Order'] = df_fanfooty_player_raw['Round'].map(round_order_dict)
df_fanfooty_player_raw

,Fanfooty Match ID,Fanfooty Match URL,Round,Year,Player ID,First Name,Surname,Team,null,DT,SC,null2,null3,null4,Kicks,Handballs,Marks,Tackles,Hitouts,Frees for,Frees against,Goals,Behinds,Not sure,Tag,Tag Notes,Tag 2,Tag 2 Notes,null5,null6,null7,null8,Position,Jumper Number,null9,null10,null11,DT own %,SC own %,AF own %,null12,AF Breakeven,null13,Contested Possessions,Clearances,Clangers,Disposal efficiency,Time on ground,Metres gained,Bench staus,Injured,Opposition_team_SC,My_team_SC,Round Order
0,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,990020.0,Andrew,Embley,WC,30.0,111.0,98,144.0,79.0,112.0,20.0,8.0,1.0,6.0,1.0,1.0,0.0,1.0,0.0,Full Time,gun,Dempsey going with him... %s from %O and %T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
1,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,230254.0,Adam,Selwood,WC,50.0,107.0,107,143.0,79.0,108.0,10.0,9.0,4.0,11.0,0.0,3.0,2.0,1.0,0.0,Full Time,hot,Tagged by Lonergan... %D and %M with %T plus %s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
2,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,200112.0,Dean,Cox,WC,27.0,99.0,118,114.0,88.0,106.0,9.0,10.0,2.0,2.0,30.0,4.0,1.0,1.0,1.0,Full Time,news,%H and %P with %s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
3,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,240016.0,Beau,Waters,WC,26.0,98.0,84,130.0,79.0,117.0,15.0,13.0,5.0,6.0,0.0,0.0,4.0,0.0,0.0,Full Time,news,%P and %M with %F... clangers and FA dampening...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
4,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,261911.0,Brad,Ebert,WC,26.0,94.0,109,121.0,70.0,96.0,12.0,9.0,3.0,6.0,0.0,1.0,0.0,1.0,0.0,Full Time,news,Matched up on Winderlich... %D and %T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136963,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1011985.0,Hugo,Ralphsmith,RI,8.0,44.0,51,36.0,30.0,42.0,6.0,2.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0,Full Time,wing,%P including %K... also %T and %M... Starting ...,NaN,NaN,NaN,NaN,NaN,NaN,Back,13.0,0.0,0.0,0.0,0.13,0.44,0.43,NaN,NaN,NaN,2.0,0.0,1.0,87.0,88.0,196.0,0.0,False,1922,1369,16
136964,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1028521.0,Luke,Trainor,RI,7.0,43.0,63,28.0,39.0,54.0,6.0,9.0,3.0,0.0,0.0,1.0,1.0,0.0,0.0,Full Time,wing,%M and %P... Starting on a wing,NaN,NaN,NaN,NaN,NaN,NaN,Back,31.0,0.0,0.0,0.0,0.00,0.00,0.00,NaN,NaN,NaN,5.0,1.0,3.0,93.0,78.0,205.0,1.0,False,1922,1369,16
136965,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1017086.0,Tom,Brown,RI,3.0,40.0,64,26.0,29.0,39.0,7.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,Full Time,guard,%P including %K... also %M and %T... Starting ...,NaN,NaN,NaN,NaN,NaN,NaN,Back,30.0,0.0,0.0,0.0,1.55,1.54,0.25,NaN,NaN,NaN,2.0,0.0,0.0,75.0,84.0,214.0,0.0,False,1922,1369,16
136966,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1023056.0,Steely,Green,RI,8.0,32.0,63,28.0,27.0,36.0,3.0,4.0,1.0,2.0,0.0,1.0,1.0,1.0,0.0,Full Time,wing,%P and %T plus %s... Rotating at half forward,NaN,NaN,NaN,NaN,NaN,NaN,Midfielder,48.0,0.0,0.0,0.0,0.18,0.07,0.22,NaN,NaN,NaN,4.0,0.0,1.0,85.0,84.0,125.0,0.0,False,1922,1369,16


## 3. Save it all to /exports

In [13]:
df_fanfooty_player_raw.to_csv("{}/{}".format(destination, file_name))
df_fanfooty_player_raw

,Fanfooty Match ID,Fanfooty Match URL,Round,Year,Player ID,First Name,Surname,Team,null,DT,SC,null2,null3,null4,Kicks,Handballs,Marks,Tackles,Hitouts,Frees for,Frees against,Goals,Behinds,Not sure,Tag,Tag Notes,Tag 2,Tag 2 Notes,null5,null6,null7,null8,Position,Jumper Number,null9,null10,null11,DT own %,SC own %,AF own %,null12,AF Breakeven,null13,Contested Possessions,Clearances,Clangers,Disposal efficiency,Time on ground,Metres gained,Bench staus,Injured,Opposition_team_SC,My_team_SC,Round Order
0,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,990020.0,Andrew,Embley,WC,30.0,111.0,98,144.0,79.0,112.0,20.0,8.0,1.0,6.0,1.0,1.0,0.0,1.0,0.0,Full Time,gun,Dempsey going with him... %s from %O and %T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
1,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,230254.0,Adam,Selwood,WC,50.0,107.0,107,143.0,79.0,108.0,10.0,9.0,4.0,11.0,0.0,3.0,2.0,1.0,0.0,Full Time,hot,Tagged by Lonergan... %D and %M with %T plus %s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
2,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,200112.0,Dean,Cox,WC,27.0,99.0,118,114.0,88.0,106.0,9.0,10.0,2.0,2.0,30.0,4.0,1.0,1.0,1.0,Full Time,news,%H and %P with %s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
3,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,240016.0,Beau,Waters,WC,26.0,98.0,84,130.0,79.0,117.0,15.0,13.0,5.0,6.0,0.0,0.0,4.0,0.0,0.0,Full Time,news,%P and %M with %F... clangers and FA dampening...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
4,3425,http://live.fanfooty.com.au/game/matchcentre.h...,R4,2010,261911.0,Brad,Ebert,WC,26.0,94.0,109,121.0,70.0,96.0,12.0,9.0,3.0,6.0,0.0,1.0,0.0,1.0,0.0,Full Time,news,Matched up on Winderlich... %D and %T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1525,1739,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136963,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1011985.0,Hugo,Ralphsmith,RI,8.0,44.0,51,36.0,30.0,42.0,6.0,2.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0,Full Time,wing,%P including %K... also %T and %M... Starting ...,NaN,NaN,NaN,NaN,NaN,NaN,Back,13.0,0.0,0.0,0.0,0.13,0.44,0.43,NaN,NaN,NaN,2.0,0.0,1.0,87.0,88.0,196.0,0.0,False,1922,1369,16
136964,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1028521.0,Luke,Trainor,RI,7.0,43.0,63,28.0,39.0,54.0,6.0,9.0,3.0,0.0,0.0,1.0,1.0,0.0,0.0,Full Time,wing,%M and %P... Starting on a wing,NaN,NaN,NaN,NaN,NaN,NaN,Back,31.0,0.0,0.0,0.0,0.00,0.00,0.00,NaN,NaN,NaN,5.0,1.0,3.0,93.0,78.0,205.0,1.0,False,1922,1369,16
136965,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1017086.0,Tom,Brown,RI,3.0,40.0,64,26.0,29.0,39.0,7.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,Full Time,guard,%P including %K... also %M and %T... Starting ...,NaN,NaN,NaN,NaN,NaN,NaN,Back,30.0,0.0,0.0,0.0,1.55,1.54,0.25,NaN,NaN,NaN,2.0,0.0,0.0,75.0,84.0,214.0,0.0,False,1922,1369,16
136966,9328,http://live.fanfooty.com.au/game/matchcentre.h...,R15,2025,1023056.0,Steely,Green,RI,8.0,32.0,63,28.0,27.0,36.0,3.0,4.0,1.0,2.0,0.0,1.0,1.0,1.0,0.0,Full Time,wing,%P and %T plus %s... Rotating at half forward,NaN,NaN,NaN,NaN,NaN,NaN,Midfielder,48.0,0.0,0.0,0.0,0.18,0.07,0.22,NaN,NaN,NaN,4.0,0.0,1.0,85.0,84.0,125.0,0.0,False,1922,1369,16


In [14]:
"{}/{}".format(destination, file_name)

'exports/scrape_20250720-213302/fanfooty_match_data_20250720-213302.csv'